# 1. Monte Carlo Prediction
### Estimates State-Value Functions from Experience

Unlike Dynamic Programming, **Monte Carlo (MC) methods** do not require a prior model of the environment's transition dynamics $P(s' | s, a)$ or reward function $R(s, a)$. They learn purely from **sampled episodes of experience**.

#### Mathematical Foundation:
1. **Discounted Return from time $t$**:
   $$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots + \gamma^{T-t-1} R_T = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1}$$
2. **Value Function as Expected Return**:
   $$V^\pi(s) = \mathbb{E}_\pi [G_t \mid S_t = s]$$
3. **Law of Large Numbers**:
   By the law of large numbers, the empirical average of observed returns converges to the expected return as the number of sampled visits to state $s$ approaches infinity:
   $$V(s) = \frac{1}{N(s)} \sum_{i=1}^{N(s)} G_i(s) \xrightarrow{N(s) \to \infty} V^\pi(s)$$


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


In [ ]:
# Step 2: Define a Pure Python Blackjack Environment (Sutton & Barto Ch. 5)
# State: (player_sum: 12..21, dealer_showing_card: 1..10, usable_ace: bool)
# Actions: 0 = Stick, 1 = Hit

class BlackjackEnv:
    def __init__(self):
        self.actions = [0, 1]  # 0: Stick, 1: Hit
        
    def _draw_card(self):
        card = np.random.randint(1, 14)
        return min(card, 10)  # Face cards (J, Q, K) have value 10; Ace is 1
        
    def _draw_hand(self):
        return [self._draw_card(), self._draw_card()]
        
    def _get_hand_value(self, hand):
        val = sum(hand)
        has_ace = 1 in hand
        if has_ace and val + 10 <= 21:
            return val + 10, True  # Usable Ace
        return val, False
        
    def reset(self, start_state=None, start_action=None):
        if start_state is not None:
            player_sum, dealer_card, usable_ace = start_state
            self.player_hand = [1, player_sum - 11] if usable_ace else [player_sum // 2, player_sum - player_sum // 2]
            self.dealer_hand = [dealer_card, self._draw_card()]
        else:
            self.player_hand = self._draw_hand()
            while self._get_hand_value(self.player_hand)[0] < 12:
                self.player_hand.append(self._draw_card())
            self.dealer_hand = self._draw_hand()
            
        p_val, usable_ace = self._get_hand_value(self.player_hand)
        return (p_val, self.dealer_hand[0], usable_ace)
        
    def step(self, action):
        if action == 1:  # Hit
            self.player_hand.append(self._draw_card())
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            if p_val > 21:
                return (p_val, self.dealer_hand[0], usable_ace), -1.0, True  # Bust
            return (p_val, self.dealer_hand[0], usable_ace), 0.0, False
        else:  # Stick: Dealer plays
            while True:
                d_val, _ = self._get_hand_value(self.dealer_hand)
                if d_val >= 17:
                    break
                self.dealer_hand.append(self._draw_card())
                
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            d_val, _ = self._get_hand_value(self.dealer_hand)
            
            if d_val > 21 or p_val > d_val:
                reward = 1.0  # Player wins
            elif p_val == d_val:
                reward = 0.0  # Draw
            else:
                reward = -1.0 # Dealer wins
                
            return (p_val, self.dealer_hand[0], usable_ace), reward, True

env = BlackjackEnv()
print("Blackjack environment initialized.")


In [ ]:
# Step 3: Monte Carlo Prediction Algorithm
def generate_episode(env, policy):
    episode = []
    state = env.reset()
    while True:
        action = policy(state)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if done:
            break
    return episode

def mc_prediction(env, policy, num_episodes=50000, gamma=1.0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)
    V = defaultdict(float)
    
    for i in range(1, num_episodes + 1):
        episode = generate_episode(env, policy)
        states_in_episode = [x[0] for x in episode]
        
        G = 0.0
        # Backward pass from terminal step to compute returns
        for t in reversed(range(len(episode))):
            s_t, a_t, r_t = episode[t]
            G = gamma * G + r_t
            
            # First visit check
            if s_t not in states_in_episode[:t]:
                returns_sum[s_t] += G
                returns_count[s_t] += 1
                V[s_t] = returns_sum[s_t] / returns_count[s_t]
                
    return V

# Simple policy: Stick if player sum >= 20, else Hit
simple_policy = lambda s: 0 if s[0] >= 20 else 1
V_pred = mc_prediction(env, simple_policy, num_episodes=50000)
print(f"Total states evaluated: {len(V_pred)}")


In [ ]:
# Step 4: Visualize State-Value Estimates Surface (No Usable Ace vs Usable Ace)
fig = plt.figure(figsize=(14, 6))

for idx, (usable_ace, title) in enumerate([(False, 'No Usable Ace'), (True, 'Usable Ace')]):
    ax = fig.add_subplot(1, 2, idx + 1, projection='3d')
    player_range = np.arange(12, 22)
    dealer_range = np.arange(1, 11)
    X, Y = np.meshgrid(dealer_range, player_range)
    Z = np.array([[V_pred.get((p, d, usable_ace), 0.0) for d in dealer_range] for p in player_range])
    
    surf = ax.plot_surface(X, Y, Z, cmap='coolwarm', edgecolor='k', linewidth=0.5, alpha=0.8)
    ax.set_xlabel('Dealer Showing')
    ax.set_ylabel('Player Sum')
    ax.set_zlabel('Value V(s)')
    ax.set_title(title)
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)

plt.suptitle("Monte Carlo Prediction: State Values for Simple Policy", fontsize=14)
plt.tight_layout()
plt.show()


### Summary
- Estimates $V(s)$ by averaging sample returns across multiple simulated episodes.
- No model of $P(s' | s, a)$ needed (model-free).
- Estimates for each state are independent; updates do not bootstrap from other states' estimates.
